Found XSRF token path from HAR file of a request to trains api: https://eticket.railway.uz/api/v3/handbook/trains/list


In [81]:
import json
import requests

with open("full_request.har", encoding="utf-8") as f:
    har = json.load(f)

for entry in har["log"]["entries"]:
    for header in entry["response"]["headers"]:
        if "xsrf" in header["value"].lower() or "xsrf" in header["name"].lower():
            print(entry["request"]["method"], entry["request"]["url"])
            print(f"  {header['name']}: {header['value']}")
            print()
            break

GET https://eticket.railway.uz/api/v1/csrf-token
  set-cookie: XSRF-TOKEN=b78ca210-a9c4-4c43-a9b9-bf95d520109b; Path=/



Testing token extraction.

In [82]:
session = requests.Session()
session.get("https://eticket.railway.uz/api/v1/csrf-token")
csrf_token = session.cookies.get("XSRF-TOKEN")
print(csrf_token)

293eea8e-41a3-4cc3-be36-b53e1f1a3b4c


Now make a test request to trains API with the token.

In [83]:
session = requests.Session()

# 1. Get CSRF token
session.get("https://eticket.railway.uz/api/v1/csrf-token")
csrf_token = session.cookies.get("XSRF-TOKEN")

# 2. Query trains
resp = session.post(
    "https://eticket.railway.uz/api/v3/handbook/trains/list",
    headers={
        "X-XSRF-TOKEN": csrf_token,
        "Content-Type": "application/json",
        "Accept": "application/json",
        "Accept-Language": "ru",
        "Referer": "https://eticket.railway.uz/ru/home",
        "Origin": "https://eticket.railway.uz",
        "device-type": "BROWSER",
    },
    json={
        "directions": {
            "forward": {
                "date": "2026-04-29",
                "depStationCode": "2900000",
                "arvStationCode": "2900700",
            }
        }
    },
)

print(resp.status_code)
print(json.dumps(resp.json(), indent=2, ensure_ascii=False))

KeyboardInterrupt: 

The brand field is cleanest binning key. Across both responses:

Afrosiyob → type varies inconsistently: "Высокоскоростной", "скрст", "СКРСТ" — but brand is always "Afrosiyob"

Sharq → brand: "Sharq", type "СК"

Nasaf → brand: "Nasaf", type "СК"

Everything else → brand is "Пассажирский", "скорый", "Cкорый" (note the Latin C in one of them!)

Departure stations. "depStationName": 
"ТАШКЕНТ": "2900000",
"САМАРКАНД": "2900700"
"БУХАРА": "2900800"

In [ ]:
def request_trains(date, departure="2900000", arrival="2900700"):
    session = requests.Session()
    session.get("https://eticket.railway.uz/api/v1/csrf-token")
    csrf_token = session.cookies.get("XSRF-TOKEN")

    resp = session.post(
        "https://eticket.railway.uz/api/v3/handbook/trains/list",
        headers={
            "X-XSRF-TOKEN": csrf_token,
            "Content-Type": "application/json",
            "Accept": "application/json",
            "Accept-Language": "ru",
            "Referer": "https://eticket.railway.uz/ru/home",
            "Origin": "https://eticket.railway.uz",
            "device-type": "BROWSER",
        },
        json={
            "directions": {
                "forward": {
                    "date": date,
                    "depStationCode": departure,
                    "arvStationCode": arrival,
                }
            }
        },
    )

    trains = resp.json()["data"]["directions"]["forward"]["trains"]
    return trains


In [ ]:
# Interesting stations, brands, classes, etc. can be hardcoded:
stations = {"Tashkent": "2900000", "Samarkand": "2900700", "Bukhara": "2900800"}
brands = ["Afrosiyob", "Sharq", "Nasaf"] # There are more, I typically care only about Afrosiyob
classes = {
    "Afrosiyob": {"2Е": "e", "1С": "b"},
    "Sharq": {"2В": "e", "1С": "b", "1В": "vip"},
    "Nasaf": {"2В": "e"},
} # class codes differ by brand, so we need to specify them separately.

# Filter out by brand and availability
trains = request_trains(departure=stations["Tashkent"], arrival=stations["Samarkand"], date="2026-05-20")
for t in trains:
    if t["brand"] in brands and len(t["cars"]) > 0:
        for car in t["cars"]:
            #print(car)
            for tariff in car["tariffs"]:
                brand = t["brand"]
                num = t["number"]
                seats = tariff["freeSeats"]
                cls = classes[brand].get(tariff["classServiceType"], "unknown")
                code = tariff["classServiceType"]
                price = tariff["tariff"]
                print(f"{brand} {num} has {seats} {cls} ({code}) seats available at {price} UZS")


Afrosiyob 768Ф has 2 business (1С) seats available at 455000 UZS
Afrosiyob 768Ф has 7 economic (2Е) seats available at 311000 UZS
Afrosiyob 766Ф has 7 business (1С) seats available at 455000 UZS
Afrosiyob 766Ф has 28 economic (2Е) seats available at 311000 UZS
Afrosiyob 770Ф has 1 economic (2Е) seats available at 311000 UZS
Sharq 710Ф has 116 business (1С) seats available at 300560 UZS
Sharq 710Ф has 10 vip (1В) seats available at 566450 UZS
Sharq 710Ф has 105 economic (2В) seats available at 200060 UZS
Nasaf 716Ф has 143 economic (2В) seats available at 200060 UZS
Sharq 712Ф has 10 business (1С) seats available at 300560 UZS
Sharq 712Ф has 206 economic (2В) seats available at 200060 UZS
Sharq 712Ф has 170 economic (2В) seats available at 200060 UZS


In [ ]:
import argparse
from datetime import datetime, date, time
import random
import time as time_module

parser = argparse.ArgumentParser(
    prog='Railway Ticket Alert',
    description='Sends a request to the railway API every n seconds and alerts you if requested tickets are available',
    epilog='')
parser.add_argument('-w', '--when', type=date.fromisoformat, required=True, help='Date of departure in ISO  YYYY-MM-DD format')
parser.add_argument('-dep', '--departure', type=str, required=True, help='Departure station')
parser.add_argument('-arr', '--arrival', type=str, required=True, help='Arrival station')
parser.add_argument('-af', '--afrosiyob', type=str, required=False, default="ae", help='Include Afrosiyob trains (economic: e, business: b, together: aeb)')
parser.add_argument('-sh', '--sharq', type=str, required=False, default="s", help='Include Sharq/Nasaf trains (economic: e, business: b, together: seb)')
parser.add_argument('--from-time', type=time.fromisoformat, required=False, default="00:00", help='Start of departure time window in HH:MM format')
parser.add_argument('--to-time', type=time.fromisoformat, required=False, default="23:59", help='End of departure time window in HH:MM format')
parser.add_argument('-i', '--interval', type=int, required=False, default=60, help='Interval between requests in seconds')

args = parser.parse_args()

when = args.when
departure = stations.get(args.departure)
arrival = stations.get(args.arrival)


usage: Railway Ticket Alert [-h] -w WHEN -dep DEPARTURE -arr ARRIVAL
                            [-af AFROSIYOB] [-sh SHARQ]
                            [--from-time FROM_TIME] [--to-time TO_TIME]
                            [-i INTERVAL]
Railway Ticket Alert: error: argument --from-time: invalid fromisoformat value: 'c:\\Users\\David\\AppData\\Roaming\\jupyter\\runtime\\kernel-v342710d1944020e3de3304d6aad75caf4d30dd75e.json'


SystemExit: 2

c:\Users\David\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [85]:
import argparse
import requests
import json
from datetime import datetime, date, time
import random
import time as time_module

departure = '2900000'  # Tashkent
arrival = '2900700'    # Samarkand
when = "2026-05-20"
af = 'ae'
sh = 'sb'
from_time = time.fromisoformat("00:00")
to_time = time.fromisoformat("23:59")
interval = 60

# Stations, brands, classes, etc. of interest can be hardcoded:
stations = {"tashkent": "2900000", "samarkand": "2900700", "bukhara": "2900800"}
brands = ["Afrosiyob", "Sharq", "Nasaf"] # There are more, I typically care only about Afrosiyob
classes = {
    "Afrosiyob": {"2Е": "e", "1С": "b"},
    "Sharq": {"2В": "e", "1С": "b", "1В": "v"},
    "Nasaf": {"2В": "e"},
} # class codes differ by brand, so we need to specify them separately. 


def request_trains(date, departure="2900000", arrival="2900700"):
    session = requests.Session()
    session.get("https://eticket.railway.uz/api/v1/csrf-token")
    csrf_token = session.cookies.get("XSRF-TOKEN")
    resp = session.post(
        "https://eticket.railway.uz/api/v3/handbook/trains/list",
        headers={
            "X-XSRF-TOKEN": csrf_token,
            "Content-Type": "application/json",
            "Accept": "application/json",
            "Accept-Language": "ru",
            "Referer": "https://eticket.railway.uz/ru/home",
            "Origin": "https://eticket.railway.uz",
            "device-type": "BROWSER",
        },
        json={
            "directions": {
                "forward": {
                    "date": date,
                    "depStationCode": departure,
                    "arvStationCode": arrival,
                }
            }
        },
    )
    trains = resp.json()["data"]["directions"]["forward"]["trains"]
    return trains

def train_fits(t: dict, af: str, sh: str, from_time: time, to_time: time) -> bool:
    if t["brand"] == 'Afrosiyob' and t["cls"] in af and from_time <= t["dep_time"] <= to_time:
        return True
    if (t["brand"] == 'Sharq' or t["brand"] == 'Nasaf') and t["cls"] in sh and from_time <= t["dep_time"] <= to_time:
        return True
    return False

def filter_trains(trains, af, sh, from_time, to_time):
    
    ## Flatten trains data structure
    inline_trains = []
    for t in trains:
        if len(t["cars"]) > 0:
            for car in t["cars"]:
                for tariff in car["tariffs"]:
                    brand = t["brand"]
                    t_num = t["number"]
                    seats = tariff["freeSeats"]
                    cls = classes.get(brand, {}).get(tariff["classServiceType"], "?") # get class code (e,b,v), default to "?" if not found
                    code = tariff["classServiceType"]
                    price = tariff["tariff"]
                    dep_time = datetime.strptime(t['departureDate'], "%d.%m.%Y %H:%M").time()
                    
                    inline_trains.append({"brand": brand, "t_num": t_num, "dep_time": dep_time, "seats": seats, "cls": cls, "code": code, "price": price})
                    #print(f"{brand} {t_num} departing at {dep_time} has {seats} {cls} ({code}) seats available at {price} UZS")

    # Filtering departure time and class
    filtered_trains = []
    for t in inline_trains:
        if train_fits(t, af, sh, from_time, to_time):
            filtered_trains.append(t)
            #print(f"{t['brand']} {t['t_num']} departing at {t['dep_time']} has {t['seats']} {t['cls']} ({t['code']}) seats available at {t['price']} UZS")
    return filtered_trains

def ticket_alert(filtered_trains):
    if filtered_trains:
        play_sound()  # Implement this function to play a sound alert
        for t in filtered_trains:
            print(f"Alert: {t['brand']} {t['t_num']} departing at {t['dep_time']} has {t['seats']} {t['cls']} ({t['code']}) seats available at {t['price']} UZS")

def play_sound():
    import winsound
    # Play a system sound
    for _ in range(3):
        winsound.PlaySound("SystemExclamation", winsound.SND_ALIAS)


while True:
    trains = request_trains(departure=departure, arrival=arrival, date=when)
    filtered_trains = filter_trains(trains, af, sh, from_time, to_time)
    ticket_alert(filtered_trains)
    time_module.sleep(interval + random.uniform(-5, 5))  # add some jitter to avoid looking like a bot

Alert: Afrosiyob 768Ф departing at 08:00:00 has 7 e (2Е) seats available at 311000 UZS
Alert: Afrosiyob 766Ф departing at 07:30:00 has 24 e (2Е) seats available at 311000 UZS
Alert: Afrosiyob 770Ф departing at 08:30:00 has 1 e (2Е) seats available at 311000 UZS
Alert: Sharq 710Ф departing at 08:37:00 has 116 b (1С) seats available at 300560 UZS
Alert: Sharq 712Ф departing at 20:32:00 has 10 b (1С) seats available at 300560 UZS
Alert: Afrosiyob 768Ф departing at 08:00:00 has 7 e (2Е) seats available at 311000 UZS
Alert: Afrosiyob 766Ф departing at 07:30:00 has 24 e (2Е) seats available at 311000 UZS
Alert: Afrosiyob 770Ф departing at 08:30:00 has 1 e (2Е) seats available at 311000 UZS
Alert: Sharq 710Ф departing at 08:37:00 has 116 b (1С) seats available at 300560 UZS
Alert: Sharq 712Ф departing at 20:32:00 has 10 b (1С) seats available at 300560 UZS
Alert: Afrosiyob 768Ф departing at 08:00:00 has 7 e (2Е) seats available at 311000 UZS
Alert: Afrosiyob 766Ф departing at 07:30:00 has 24 

KeyboardInterrupt: 